In [2]:
import os
import sys
from PIL import Image
import pyocr
import pyocr.builders
from pdf2image import convert_from_path

def get_tool():
    tools = pyocr.get_available_tools()
    if len(tools) == 0:
        raise RuntimeError("No OCR tool found. Install Tesseract and pyocr.")
    return tools[0]


def load_images(input_path):
    """
    Load images either from an image file or from a PDF.
    Returns a list of PIL Image objects.
    """
    ext = os.path.splitext(input_path)[1].lower()
    if ext == '.pdf':
        # Convert PDF to images
        return convert_from_path(input_path)
    else:
        # Load single image
        img = Image.open(input_path)
        return [img]


def split_into_chunks(img, rows=2, cols=4):
    """
    Split a PIL Image into rows*cols chunks.
    Returns a list of image chunks.
    """
    w, h = img.size
    chunk_w = w // cols
    chunk_h = h // rows
    chunks = []
    for r in range(rows):
        for c in range(cols):
            left = c * chunk_w
            upper = r * chunk_h
            right = (c + 1) * chunk_w if c < cols - 1 else w
            lower = (r + 1) * chunk_h if r < rows - 1 else h
            chunk = img.crop((left, upper, right, lower))
            chunks.append(chunk)
    return chunks


def ocr_chunks(chunks, tool):
    """
    Perform OCR on each image chunk and return list of text results.
    """
    texts = []
    for idx, chunk in enumerate(chunks):
        txt = tool.image_to_string(
            chunk,
            lang="eng",
            builder=pyocr.builders.TextBuilder()
        )
        texts.append((idx, txt))
    return texts


def main(input_path):
    # Initialize OCR tool
    tool = get_tool()
    print(f"Using OCR tool: {tool.get_name()}")

    # Load images
    images = load_images(input_path)
    for page_num, img in enumerate(images):
        print(f"\n--- Page {page_num + 1} ---")
        # Split into 8 chunks
        chunks = split_into_chunks(img, rows=2, cols=4)
        # OCR each chunk
        results = ocr_chunks(chunks, tool)
        # Output text per chunk
        for idx, text in results:
            print(f"\nChunk {idx + 1}:\n{text}\n")

if __name__ == "__main__":
    if len(sys.argv) < 2:
        print(f"Usage: python {sys.argv[0]} <image_or_pdf_path>")
        sys.exit(1)
    input_file = sys.argv[1]
    main(input_file)

Using OCR tool: Tesseract (sh)


OSError: [Errno 22] Invalid argument: '--f=c:\\Users\\maksi\\AppData\\Roaming\\jupyter\\runtime\\kernel-v33cf90f4c9f4b8ebfca50fd95b9d34ecc2a189809.json'